### This code is to implement SCD Type 2 using merge

In [0]:
# Assign libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable  #Important for Merge

In [0]:
# We are dealing with Patient Data. This is a existing patient data
existing_data = [
    (1, "John", "Doe", "M", 30, "2025-01-01", None, "Y"),
    (2, "Mary", "James", "F", 25, "2025-01-01", None, "Y")
]

existing_schema = StructType([
    StructField("patient_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True),
    StructField("is_current", StringType(), True)
])

dim_df = spark.createDataFrame(existing_data, existing_schema)

dim_df.show()

dim_df.write.format("delta").mode("overwrite").saveAsTable("sample_catalog.sample_schema.dim_patient")

+----------+----------+---------+------+---+----------+--------+----------+
|patient_id|first_name|last_name|gender|age|start_date|end_date|is_current|
+----------+----------+---------+------+---+----------+--------+----------+
|         1|      John|      Doe|     M| 30|2025-01-01|    NULL|         Y|
|         2|      Mary|    James|     F| 25|2025-01-01|    NULL|         Y|
+----------+----------+---------+------+---+----------+--------+----------+



In [0]:
#Incoming source data

new_data = [
    (1, "John", "Doe", "M", 31),
    (2, "Mary", "James", "F", 25),
    (3, "Sam", "Wilson", "M", 40)
]

new_schema = StructType([
    StructField("patient_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True)])

source_df = spark.createDataFrame(new_data, new_schema)

source_df.show()

+----------+----------+---------+------+---+
|patient_id|first_name|last_name|gender|age|
+----------+----------+---------+------+---+
|         1|      John|      Doe|     M| 31|
|         2|      Mary|    James|     F| 25|
|         3|       Sam|   Wilson|     M| 40|
+----------+----------+---------+------+---+



In [0]:
#Create a Dataframe using the Delta Table
delta_table = DeltaTable.forName(spark, "sample_catalog.sample_schema.dim_patient")
current_df = delta_table.toDF().filter(col("is_current") == "Y")
current_df.show()


+----------+----------+---------+------+---+----------+--------+----------+
|patient_id|first_name|last_name|gender|age|start_date|end_date|is_current|
+----------+----------+---------+------+---+----------+--------+----------+
|         1|      John|      Doe|     M| 30|2025-01-01|    NULL|         Y|
|         2|      Mary|    James|     F| 25|2025-01-01|    NULL|         Y|
+----------+----------+---------+------+---+----------+--------+----------+



In [0]:
# Identify changed records
changed_df = source_df.alias("src").join(
    current_df.alias("tgt"),
    "patient_id"
).filter(
    (col("src.first_name") != col("tgt.first_name")) |
    (col("src.last_name") != col("tgt.last_name")) |
    (col("src.gender") != col("tgt.gender")) |
    (col("src.age") != col("tgt.age"))
).select("src.*")

changed_df.show()

+----------+----------+---------+------+---+
|patient_id|first_name|last_name|gender|age|
+----------+----------+---------+------+---+
|         1|      John|      Doe|     M| 31|
+----------+----------+---------+------+---+



In [0]:
# Delta merge to update the changed records audit fields
delta_table.alias("tgt").merge(
    changed_df.alias("src"),
    "tgt.patient_id = src.patient_id"
).whenMatchedUpdate(
    set={
        "is_current": "'N'",
        "end_date": "current_date()"
    }
).execute()
        

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

+----------+----------+---------+------+---+----------+----------+----------+
|patient_id|first_name|last_name|gender|age|start_date|  end_date|is_current|
+----------+----------+---------+------+---+----------+----------+----------+
|         2|      Mary|    James|     F| 25|2025-01-01|      NULL|         Y|
|         1|      John|      Doe|     M| 30|2025-01-01|2026-05-14|         N|
+----------+----------+---------+------+---+----------+----------+----------+



In [0]:
# New records to insert
new_records_df = source_df.alias("src").join(
    delta_table.toDF().filter(col("is_current") == "Y").alias("tgt"),
    "patient_id",
    "left_anti"
)

# Merge Changed records and New freocrds and update audit fields
final_insert_df = (
    changed_df.unionByName(new_records_df)
    .withColumn("start_date", lit(current_date()).cast("String"))
    .withColumn("end_date", lit(None).cast("String"))
    .withColumn("is_current", lit("Y"))
)

In [0]:
# Insert into Delta Table
final_insert_df.write.format("delta").mode("append").saveAsTable("sample_catalog.sample_schema.dim_patient")

